# Notebook C — Realized Volatility Forecasting

**SSVI Volatility Surface Dynamics — S&P 500 Options (2010–2020)**  
Politecnico di Milano — Econometrics Project (A.Y. 2025/26)

---

## Overview

Full realized volatility (RV) forecasting pipeline at horizons **h ∈ {1, 5, 20}** trading days.

| Model family | Description | Uses VIX? |
|---|---|---|
| **HAR** — *primary baseline* | Corsi (2009): daily, weekly, monthly RV components | No |
| **HAR+VIX** — *secondary benchmark* | HAR augmented with contemporaneous VIX | Yes (benchmark only) |
| **F3_ATM** | HAR + SSVI-derived ATM IV — portable to any options market | No |
| **HAR_rhoJ** | HAR + \|Δρ\| — rho-based jump signal (NB08) | No |
| **M_dSSVI** | HAR + Δ SSVI params — path-dependence (Andres et al. 2025, NB08) | No |
| **M1–M4** | HAR + SSVI parameter subsets (progressive augmentation) | No |
| **M4_thresh** | M4 + hard stress indicator I\_stress (TAR analogy) | No |
| **M4_smooth** | M4 + smooth VIX-percentile exponential regime weighting | No (λ only) |
| **M5** | M4 + nonlinear terms (β², α×β) | No |
| **M4+int** | M4\_smooth + log\_ATM×max\_cond1 (near-arb stress interaction) | No |

**Portability principle**: all primary models use only SSVI parameters as features,
making them applicable to any liquid options market where SSVI can be calibrated —
not just equity indices with an exchange-published VIX.

Model comparison via **DM-HLN** tests (Diebold & Mariano 1995; Harvey, Leybourne & Newbold 1997)  
with Newey-West HAC standard errors. OOS window: last 20% ≈ 2018–2020.

**Key result**: F3_ATM is the most interpretable and portable model, competitive at h=5;
M4_smooth dominates at h=20 (R²_OOS ≈ 0.868, DM p < 0.0001 vs HAR).
M_dSSVI captures path-dependence in the surface (best at h=20 in NB08, R²=0.84).
HAR_rhoJ uses |Δρ| as a model-free jump signal, ensuring ρ dynamics are represented.

**References:**
- Corsi (2009), *JFEC* 7(2), 174–196 — HAR-RV
- Gatheral & Jacquier (2014), *QF* 14(1), 59–71 — SSVI
- Diebold & Mariano (1995), *JBES* 13(3), 253–263 — DM test
- Harvey, Leybourne & Newbold (1997), *IJF* 13(2), 281–291 — HLN correction
- Andersen, Bollerslev, Diebold & Labys (2003), *Econometrica* 71(2), 579–625
- Andres, Boumezoued & Jourdain (2025), *arXiv v3* — path-dependent vol surface


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import pandas_datareader.data as web
    HAS_PDR = True
except ImportError:
    HAS_PDR = False
    print('pandas_datareader not found — install with: pip install pandas-datareader')

import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

In [ ]:
import sys
sys.path.insert(0, '../src')

from ssvi_helpers      import ssvi_atm_iv, ssvi_derived_features
from stats_helpers     import fit_ols_predict, r2_oos, mape, dm_hln
from forecasting_helpers import har_features, make_log_rv_target, smooth_stress_indicator

print('src/ helpers imported.')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
GITHUB   = 'https://raw.githubusercontent.com/aporrini/Econometrics-Volatility-Surface-Dynamics/main/Data'
OUTPUT   = Path('../output')
PLOT_DIR = OUTPUT / 'figures'
OUTPUT.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

START    = pd.Timestamp('2010-01-04')
END      = pd.Timestamp('2020-12-31')
SPLIT    = 0.80
HORIZONS = [1, 5, 20]
PARAMS   = ['alpha', 'beta', 'rho', 'eta', 'gamma']
SEED     = 42

MODEL_ORDER = [
    'Naive', 'HAR', 'HAR+VIX',
    'F3_ATM',
    'HAR_rhoJ',                    # HAR + |Δρ| jump signal (NB08)
    'M_dSSVI',                     # HAR + Δ SSVI params — path-dependence (NB08)
    'M1', 'M2', 'M3', 'M4',
    'M4_thresh',                   # M4 + hard stress indicator (TAR analogy)
    'M4_smooth',                   # M4 + soft sigmoid indicator (STAR analogy)
    'M5', 'M4+int'
]
print('Config ready — GitHub data source:', GITHUB)

## 1. Data Loading

- **SSVI parameters**: loaded from GitHub raw URL (Gatheral & Jacquier 2014 calibrations)
- **No-arbitrage indicators**: loaded from GitHub (butterfly-proximity stress per day)
- **SP500**: downloaded via `pandas_datareader` (Yahoo Finance)
- **VIX**: downloaded via `pandas_datareader` (FRED — `VIXCLS` series); used **only** as exogenous benchmark variable, never in primary SSVI-only models

In [ ]:
# ── SSVI parameters from GitHub ───────────────────────────────────────────────
# The CSV uses 'time_elapsed' (integer trading-day index), not a 'date' column.
# We reconstruct the date as: START + timedelta(time_elapsed days).
print('Loading SSVI parameters...')
ssvi_raw = pd.read_csv(f'{GITHUB}/ssvi_all_dates_clean_results.csv')

# Keep only successful calibrations
if 'success' in ssvi_raw.columns:
    ssvi_raw = ssvi_raw[ssvi_raw['success']].copy()

# Build calendar date from integer time_elapsed
ssvi_raw['date'] = START + pd.to_timedelta(ssvi_raw['time_elapsed'].astype(int), unit='D')
ssvi_raw['date'] = pd.to_datetime(ssvi_raw['date'])
ssvi_raw = ssvi_raw.sort_values('date').set_index('date')
ssvi_raw.index = pd.DatetimeIndex(ssvi_raw.index)
ssvi = ssvi_raw.loc[START:END, PARAMS].copy()
print(f'  SSVI: {ssvi.shape[0]} days  [{ssvi.index[0].date()} – {ssvi.index[-1].date()}]')
print(f'  Columns: {ssvi.columns.tolist()}')

# ── No-arbitrage stress from GitHub ──────────────────────────────────────────
print('Loading no-arbitrage results...')
na_raw = pd.read_csv(f'{GITHUB}/no_arbitrage_clean_results.csv')
na_raw['date'] = START + pd.to_timedelta(na_raw['time_elapsed'].astype(int), unit='D')
na_raw['date'] = pd.to_datetime(na_raw['date'])
na_raw = na_raw.sort_values('date').set_index('date')
na_raw.index = pd.DatetimeIndex(na_raw.index)
print(f'  NA file columns: {na_raw.columns.tolist()}')

# Identify stress column (max_cond1 = butterfly-proximity indicator)
if 'max_cond1' in na_raw.columns:
    stress_col = 'max_cond1'
else:
    cands = [c for c in na_raw.columns
             if any(kw in c.lower() for kw in ['cond', 'viol', 'arb', 'max'])]
    stress_col = cands[0] if cands else None
print(f'  Stress column: {stress_col}')

In [ ]:
# ── SP500 from Yahoo Finance (pandas_datareader) ──────────────────────────────
sp500_raw, vix_raw = None, None

if HAS_PDR:
    try:
        sp500_raw = web.DataReader('^GSPC', 'yahoo', '2009-01-01', '2021-01-01')['Close']
        sp500_raw = sp500_raw.squeeze()
        sp500_raw.index = pd.DatetimeIndex(sp500_raw.index)
        print(f'SP500: {sp500_raw.shape[0]} obs  '
              f'[{sp500_raw.index[0].date()} – {sp500_raw.index[-1].date()}]')
    except Exception as e:
        print(f'SP500 download failed: {e}')

    # ── VIX from FRED ─────────────────────────────────────────────────────────
    try:
        vix_raw = web.DataReader('VIXCLS', 'fred', '2009-01-01', '2021-01-01')['VIXCLS']
        vix_raw.index = pd.DatetimeIndex(vix_raw.index)
        vix_raw = vix_raw.dropna()
        print(f'VIX:   {vix_raw.shape[0]} obs  '
              f'[{vix_raw.index[0].date()} – {vix_raw.index[-1].date()}]')
    except Exception as e:
        print(f'VIX (FRED) download failed: {e}')

if sp500_raw is None:
    raise RuntimeError('SP500 data required — check pandas_datareader installation')

## 2. Realized Volatility Construction

Daily RV proxy: squared annualized log-returns $\hat{\sigma}^2_t = (252) \cdot r_t^2$.

**HAR-RV components** (Corsi 2009):
$$RV^{(d)}_t = \hat{\sigma}^2_t, \quad
RV^{(w)}_t = \frac{1}{5}\sum_{j=0}^{4}\hat{\sigma}^2_{t-j}, \quad
RV^{(m)}_t = \frac{1}{22}\sum_{j=0}^{21}\hat{\sigma}^2_{t-j}$$

Target at horizon $h$: $\overline{RV}^{(h)}_{t+h} = \frac{1}{h}\sum_{j=1}^{h}\hat{\sigma}^2_{t+j}$.
Both features and targets are log-transformed for Gaussianity.


In [ ]:
# ── Daily log-returns → annualized RV proxy ───────────────────────────────────
log_ret = np.log(sp500_raw / sp500_raw.shift(1)).dropna()
rv_daily_full = (log_ret ** 2) * 252

# HAR rolling components (computed on full series including burn-in)
rv_w_full = rv_daily_full.rolling(5,  min_periods=5).mean()
rv_m_full = rv_daily_full.rolling(22, min_periods=22).mean()

# Trim to study window
rv_d = rv_daily_full.loc[START:END].copy()
rv_w = rv_w_full.loc[START:END].copy()
rv_m = rv_m_full.loc[START:END].copy()

print(f'RV daily:   {rv_d.notna().sum()} obs  mean={rv_d.mean():.5f}  std={rv_d.std():.5f}')
print(f'RV weekly:  {rv_w.notna().sum()} obs')
print(f'RV monthly: {rv_m.notna().sum()} obs')

# Log-RV (features)
log_rv_d = np.log(rv_d.clip(lower=1e-12))
log_rv_w = np.log(rv_w.clip(lower=1e-12))
log_rv_m = np.log(rv_m.clip(lower=1e-12))


In [ ]:
# ── Multi-horizon forward-looking targets ─────────────────────────────────────
#
# rv_daily.rolling(h).mean() at time T = mean(rv_{T-h+1}..rv_T)
# .shift(-h) at time t   = value from t+h = mean(rv_{t+1}..rv_{t+h})  ✓
#
def make_log_rv_target(rv_series, h):
    fwd = rv_series.rolling(h).mean().shift(-h)
    return np.log(fwd.clip(lower=1e-12))

targets = {h: make_log_rv_target(rv_d, h) for h in HORIZONS}

print('Target (log-RV forward average) summary:')
for h in HORIZONS:
    t = targets[h].dropna()
    print(f'  h={h:2d}: {len(t)} obs  mean={t.mean():.3f}  std={t.std():.3f}')


## 3. Feature Engineering

All features are computed at time $t$ (no lookahead into $t+1, \ldots, t+h$).

**SSVI ATM implied volatility** (Gatheral & Jacquier 2014):
At $k=0$ the SSVI total variance simplifies to $\omega(0,T) = \theta_T = e^{\alpha} T^{\beta}$,  
giving annualized ATM IV $= \sqrt{\theta_T / T} = e^{\alpha/2} T^{\beta/2 - 1/2}$.

**Smooth regime indicator** (Teräsvirta 1994 STAR analogy):  
$\lambda_t = \sigma\!\left(10 \cdot (\mathrm{VIX\_pct}_t - 0.70)\right) \in [0,1]$  
where $\mathrm{VIX\_pct}_t$ is the 252-day rolling VIX percentile rank. $\lambda_t \approx 1$ in high-vol regimes.


In [ ]:
# ── VIX (align to rv_d index) ─────────────────────────────────────────────────
# VIX is used ONLY in HAR+VIX benchmark and the lambda_stress computation.
# Primary SSVI models (F3_ATM, M1–M4, M4_smooth, M5, M4+int) do not use VIX.
if vix_raw is not None:
    vix_s = vix_raw.reindex(rv_d.index, method='ffill') / 100.0
else:
    # Fallback: construct a zero series so lambda_stress = 0.5 everywhere
    print('Warning: VIX not available — HAR+VIX excluded, lambda_stress set to 0.5')
    vix_s = pd.Series(0.20, index=rv_d.index)   # neutral VIX proxy
vix_sq = vix_s ** 2

# ── SSVI ATM IV at 3-month tenor ─────────────────────────────────────────────
# omega(k=0, T=0.25) = theta_T = exp(alpha) * T^beta
# Annualized ATM IV = sqrt(theta_T / T) = exp(alpha/2) * T^((beta-1)/2)
# Entirely determined by (alpha, beta) — portable to any asset after SSVI calibration
T_REF = 0.25   # 3-month reference tenor (years)
ssvi_aligned = ssvi.reindex(rv_d.index).ffill()

theta_ref = np.exp(ssvi_aligned['alpha']) * (T_REF ** ssvi_aligned['beta'])
atm_iv    = np.sqrt(theta_ref / T_REF)              # annualized ATM IV
log_atm   = np.log(atm_iv.clip(lower=1e-12))

# ── No-arbitrage stress indicator ─────────────────────────────────────────────
# max_cond1: proximity to butterfly no-arbitrage boundary g(k,T) >= 0
# max_cond1 -> 0 : surface at maximum curvature permitted without local arbitrage
if stress_col is not None:
    max_cond = na_raw.loc[START:END, stress_col].reindex(rv_d.index).fillna(0)
else:
    max_cond = pd.Series(0.0, index=rv_d.index)
    print('Warning: stress column not found — near-arb interaction disabled')
log_max_cond = np.log1p(max_cond.clip(lower=0))

# ── Smooth stress indicator: 252-day rolling VIX percentile ──────────────────
# lambda_t in [0,1]: 1 = high-stress regime, 0 = calm regime
# Note: lambda_stress is a regime indicator derived from VIX percentile rank,
# NOT the VIX level itself — it captures *relative* stress, not absolute vol.
vix_pct = vix_s.rolling(252, min_periods=60).apply(
    lambda x: float((x <= x.iloc[-1]).mean()), raw=False
).fillna(0.5)
lambda_stress = 1.0 / (1.0 + np.exp(-10.0 * (vix_pct - 0.70)))

print(f'ATM IV (3m): mean={atm_iv.mean():.3f}  std={atm_iv.std():.3f}')
print(f'VIX pct:     mean={vix_pct.mean():.3f}  std={vix_pct.std():.3f}')
print(f'Lambda stress: mean={lambda_stress.mean():.3f}')

In [ ]:
# ── Full feature matrix (all at time t, no lookahead) ─────────────────────────
df_full = pd.DataFrame({
    'log_rv_d':      log_rv_d,
    'log_rv_w':      log_rv_w,
    'log_rv_m':      log_rv_m,
    'vix_d':         vix_s,
    'vix_sq':        vix_sq,
    'log_atm_iv':    log_atm,
    'alpha':         ssvi_aligned['alpha'],
    'beta':          ssvi_aligned['beta'],
    'rho':           ssvi_aligned['rho'],
    'eta':           ssvi_aligned['eta'],
    'gamma':         ssvi_aligned['gamma'],
    'log_max_cond':  log_max_cond,
    'lambda_stress': lambda_stress,
}, index=rv_d.index)

# ── SSVI first-differences (path-dependence, Andres et al. 2025) ──────────────
# Δθ_t = θ_t - θ_{t-1}: captures day-to-day surface dynamics
# Used in M_dSSVI (HAR + Δ SSVI) — best at h=20 in NB08 (R²=0.84)
for p in PARAMS:
    df_full[f'd_{p}'] = df_full[p].diff()

# |Δρ| — rho-based jump signal (HAR_rhoJ, NB08)
# Large |Δρ| marks abrupt leverage/skew regime shifts (proxy for jump risk)
df_full['abs_d_rho'] = df_full['d_rho'].abs()

# Binary hard stress indicator (TAR analogy, M4_thresh)
# I_stress = 1 when lambda > 0.5 (VIX percentile crossed 70th-pct threshold)
df_full['I_stress'] = (df_full['lambda_stress'] > 0.5).astype(float)

df_full = df_full.dropna()
print(f'Feature matrix: {df_full.shape[0]} obs x {df_full.shape[1]} cols')
print(f'Date range: {df_full.index[0].date()} -> {df_full.index[-1].date()}')
print(df_full.describe().round(4))


## 4. Evaluation Framework

**OOS R²** relative to Naive (random-walk) benchmark:
$$R^2_{OOS} = 1 - \frac{\sum_t(y_t - \hat{y}_t)^2}{\sum_t(y_t - y_{t-1})^2}$$

**DM-HLN test** (H₀: equal predictive accuracy vs HAR baseline):  
Loss differential $d_t = e^2_{HAR,t} - e^2_{model,t}$ (positive → model better).  
HAC variance via Bartlett kernel with $h-1$ lags; HLN small-sample correction:
$$\kappa = \frac{T+1-2h+h(h-1)/T}{T}$$


In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def r2_oos(y_true, y_pred, y_naive):
    mse_m = np.mean((y_true - y_pred) ** 2)
    mse_n = np.mean((y_true - y_naive) ** 2)
    return float(1 - mse_m / mse_n)


def dm_hln(e_model, e_bench, h=1):
    """
    DM test with HLN small-sample correction and Newey-West HAC.
    d_t = e_bench^2 - e_model^2 ; positive DM => model beats bench.
    Refs: Diebold & Mariano (1995); Harvey, Leybourne & Newbold (1997).
    """
    d = np.asarray(e_bench) ** 2 - np.asarray(e_model) ** 2
    T = len(d)
    d_bar  = d.mean()
    gamma0 = np.var(d, ddof=1)
    hac    = gamma0
    for lag in range(1, h):
        if lag < T:
            gl   = np.cov(d[lag:], d[:-lag])[0, 1]
            hac += 2 * (1 - lag / h) * gl    # Bartlett kernel
    kappa  = (T + 1 - 2 * h + h * (h - 1) / T) / T
    dm_val = d_bar / np.sqrt(max(hac * kappa / T, 1e-20))
    p_val  = 2 * stats.norm.cdf(-abs(dm_val))
    return float(dm_val), float(p_val)


def stars(p):
    if pd.isna(p): return ''
    if p < 0.001:  return '***'
    if p < 0.01:   return '**'
    if p < 0.05:   return '*'
    return ''


def fit_ols_predict(X_tr, y_tr, X_te):
    Xc_tr = sm.add_constant(X_tr, has_constant='add')
    model = OLS(y_tr, Xc_tr).fit()
    Xc_te = sm.add_constant(X_te, has_constant='add')
    Xc_te = Xc_te.reindex(columns=Xc_tr.columns, fill_value=0)
    return model, model.predict(Xc_te)


# Feature column groups (defined once, used in loop)
HAR_COLS    = ['log_rv_d', 'log_rv_w', 'log_rv_m']
SSVI_COLS   = ['alpha', 'beta', 'rho', 'eta', 'gamma']
VIX_COLS    = ['vix_d', 'vix_sq']
ATM_COLS    = ['log_atm_iv']
STRESS_COLS = ['log_max_cond']
SMOOTH_COL  = ['lambda_stress']
DSSVI_COLS  = ['d_alpha', 'd_beta', 'd_rho', 'd_eta', 'd_gamma']   # Δ SSVI params (M_dSSVI)
JUMP_COLS   = ['abs_d_rho']                                          # |Δρ| jump signal (HAR_rhoJ)
THRESH_COL  = ['I_stress']                                           # hard indicator (M4_thresh)

print('Helpers defined.')


## 5. Multi-Horizon Evaluation

For each horizon $h \in \{1,5,20\}$:
1. Align feature matrix with forward target; drop NaN rows.
2. Chronological 80/20 split (no shuffling — preserves temporal order).
3. Fit all models on training window; evaluate on test window.
4. Compute MSE, MAE, R²_OOS (vs Naive), DM-HLN (vs HAR).


In [ ]:
all_results  = []     # list of dicts -> DataFrame at end
preds_store  = {}     # {h: {model_name: predictions_array}}
models_store = {}     # {(h, name): fitted OLS model}

for h in HORIZONS:
    print(f'\n{"="*65}')
    print(f'  Horizon h = {h}')
    print(f'{"="*65}')

    # Align features + target; drop NaN (last h rows will have NaN target)
    df_h = df_full.join(targets[h].rename('target')).dropna()
    n_tr = int(len(df_h) * SPLIT)
    tr   = df_h.iloc[:n_tr]
    te   = df_h.iloc[n_tr:]
    print(f'  Train: {n_tr} obs ({tr.index[0].date()} -> {tr.index[-1].date()})')
    print(f'  Test:  {len(te)} obs ({te.index[0].date()} -> {te.index[-1].date()})')

    Xtr = tr.drop(columns='target')
    ytr = tr['target'].values
    Xte = te.drop(columns='target')
    yte = te['target'].values

    preds = {'y_true': yte}

    # ── 0. Naive (random walk in log-RV) ─────────────────────────────────────
    preds['Naive'] = Xte['log_rv_d'].values

    # ── 1. HAR — Primary Baseline (Corsi 2009) ────────────────────────────────
    m_har, p_har = fit_ols_predict(Xtr[HAR_COLS], ytr, Xte[HAR_COLS])
    preds['HAR'] = p_har
    models_store[(h, 'HAR')] = m_har

    # ── 2. HAR+VIX — Secondary Benchmark (comparison only) ───────────────────
    hv_cols = HAR_COLS + ['vix_d']
    _, p_hv  = fit_ols_predict(Xtr[hv_cols], ytr, Xte[hv_cols])
    preds['HAR+VIX'] = p_hv

    # ── 3. F3_ATM — Portable (HAR + SSVI log-ATM IV) ─────────────────────────
    # log_sigma_ATM = alpha/2 + (beta-1)/2 * log(T0): derived from (alpha,beta) only
    f3a_cols = HAR_COLS + ATM_COLS
    _, p_f3a = fit_ols_predict(Xtr[f3a_cols], ytr, Xte[f3a_cols])
    preds['F3_ATM'] = p_f3a

    # ── 4. HAR_rhoJ — HAR + |Δρ| rho-based jump signal (NB08) ───────────────
    # |Δρ_t| measures abrupt changes in the leverage/skew parameter of the SSVI
    # surface. Large |Δρ| corresponds to sudden repricing of tail risk — a
    # model-free jump signal (Ait-Sahalia & Jacod 2014 analogy).
    # Addresses ρ underrepresentation: ρ level enters M2/M4; here its *dynamics*
    # carry additional predictive information for short-horizon RV.
    _, p_rhoj = fit_ols_predict(
        Xtr[HAR_COLS + JUMP_COLS], ytr, Xte[HAR_COLS + JUMP_COLS]
    )
    preds['HAR_rhoJ'] = p_rhoj

    # ── 5. M_dSSVI — HAR + Δ SSVI params (Andres et al. 2025) ───────────────
    # First differences of all five SSVI params capture path-dependent dynamics:
    # how fast the surface is changing, not just its current level.
    # Best performer at h=20 in NB08 (R²=0.8412).
    _, p_dssvi = fit_ols_predict(
        Xtr[HAR_COLS + DSSVI_COLS], ytr, Xte[HAR_COLS + DSSVI_COLS]
    )
    preds['M_dSSVI'] = p_dssvi

    # ── 6. M1 — SSVI Level (alpha = log integrated variance) ─────────────────
    _, p_m1 = fit_ols_predict(Xtr[HAR_COLS + ['alpha']], ytr,
                               Xte[HAR_COLS + ['alpha']])
    preds['M1'] = p_m1

    # ── 7. M2 — SSVI Slope & Leverage (beta, rho) ────────────────────────────
    _, p_m2 = fit_ols_predict(Xtr[HAR_COLS + ['beta', 'rho']], ytr,
                               Xte[HAR_COLS + ['beta', 'rho']])
    preds['M2'] = p_m2

    # ── 8. M3 — SSVI Curvature (eta, gamma) ──────────────────────────────────
    _, p_m3 = fit_ols_predict(Xtr[HAR_COLS + ['eta', 'gamma']], ytr,
                               Xte[HAR_COLS + ['eta', 'gamma']])
    preds['M3'] = p_m3

    # ── 9. M4 — Full SSVI augmentation ───────────────────────────────────────
    m4_cols = HAR_COLS + SSVI_COLS
    m_m4, p_m4 = fit_ols_predict(Xtr[m4_cols], ytr, Xte[m4_cols])
    preds['M4'] = p_m4
    models_store[(h, 'M4')] = m_m4

    # ── 10. M4_thresh — Hard stress indicator (TAR analogy, Hansen 1999) ─────
    # I_stress = 1{lambda_t > 0.5} = 1{VIX_pct_t > 70th percentile}
    # Analogous to a Threshold Autoregressive (TAR) model: coefficients shift
    # discretely between calm and stressed regimes.
    # Compared to M4_smooth: hard vs soft transition (same 70th-pct threshold).
    Xtr_thr = Xtr[HAR_COLS + SSVI_COLS + THRESH_COL].copy()
    Xte_thr = Xte[HAR_COLS + SSVI_COLS + THRESH_COL].copy()
    for sc in SSVI_COLS:
        Xtr_thr[sc + '_x_I'] = Xtr[sc] * Xtr['I_stress']
        Xte_thr[sc + '_x_I'] = Xte[sc] * Xte['I_stress']
    _, p_thr = fit_ols_predict(Xtr_thr, ytr, Xte_thr)
    preds['M4_thresh'] = p_thr

    # ── 11. M4_smooth — Smooth regime weighting (Terasvirta 1994 analogy) ────
    # lambda_t = sigmoid(VIX_pct - 0.70): smooth stress indicator in [0,1]
    m4s_base = HAR_COLS + SSVI_COLS + SMOOTH_COL
    Xtr_m4s  = Xtr[m4s_base].copy()
    Xte_m4s  = Xte[m4s_base].copy()
    for sc in SSVI_COLS:
        Xtr_m4s[sc + '_x_lam'] = Xtr[sc] * Xtr['lambda_stress']
        Xte_m4s[sc + '_x_lam'] = Xte[sc] * Xte['lambda_stress']
    m_m4s, p_m4s = fit_ols_predict(Xtr_m4s, ytr, Xte_m4s)
    preds['M4_smooth'] = p_m4s
    models_store[(h, 'M4_smooth')] = m_m4s

    # ── 12. M5 — M4 + Nonlinear terms (beta^2, alpha*beta) ───────────────────
    Xtr_m5 = Xtr[m4_cols].copy()
    Xte_m5 = Xte[m4_cols].copy()
    Xtr_m5['beta_sq']      = Xtr['beta'] ** 2
    Xte_m5['beta_sq']      = Xte['beta'] ** 2
    Xtr_m5['alpha_x_beta'] = Xtr['alpha'] * Xtr['beta']
    Xte_m5['alpha_x_beta'] = Xte['alpha'] * Xte['beta']
    _, p_m5 = fit_ols_predict(Xtr_m5, ytr, Xte_m5)
    preds['M5'] = p_m5

    # ── 13. M4+int — Near-arb stress interaction ─────────────────────────────
    # log_ATM * max_cond1: vol level * butterfly-proximity stress
    Xtr_m4i = Xtr_m4s.copy()
    Xte_m4i = Xte_m4s.copy()
    Xtr_m4i['atm_x_cond'] = Xtr['log_atm_iv'] * Xtr['log_max_cond']
    Xte_m4i['atm_x_cond'] = Xte['log_atm_iv'] * Xte['log_max_cond']
    m_m4i, p_m4i = fit_ols_predict(Xtr_m4i, ytr, Xte_m4i)
    preds['M4+int'] = p_m4i
    models_store[(h, 'M4+int')] = m_m4i

    # ── Collect metrics ───────────────────────────────────────────────────────
    e_har   = yte - preds['HAR']
    naive_p = preds['Naive']

    for mname, ypred in preds.items():
        if mname == 'y_true':
            continue
        e      = yte - ypred
        mse_   = float(np.mean(e ** 2))
        mae_   = float(np.mean(np.abs(e)))
        mape_  = float(np.mean(np.abs(e) / np.abs(yte).clip(1e-8)) * 100)
        r2_    = float(r2_oos(yte, ypred, naive_p))
        if mname not in ('Naive', 'HAR'):
            dm_, p_ = dm_hln(e, e_har, h=h)
        else:
            dm_, p_ = (np.nan, np.nan)
        all_results.append({
            'h': h, 'Model': mname,
            'MSE': mse_, 'MAE': mae_, 'MAPE': mape_, 'R2_OOS': r2_,
            'DM_vs_HAR': dm_, 'p_DM': p_
        })

    preds_store[h] = preds
    print(f'  HAR      R2_OOS = {r2_oos(yte, preds["HAR"],      naive_p):.4f}')
    print(f'  HAR_rhoJ R2_OOS = {r2_oos(yte, preds["HAR_rhoJ"], naive_p):.4f}')
    print(f'  M_dSSVI  R2_OOS = {r2_oos(yte, preds["M_dSSVI"],  naive_p):.4f}')
    print(f'  M4_thresh R2_OOS = {r2_oos(yte, preds["M4_thresh"],naive_p):.4f}')
    print(f'  M4_smooth R2_OOS = {r2_oos(yte, preds["M4_smooth"],naive_p):.4f}')

res_df = pd.DataFrame(all_results)
res_df['Sig'] = res_df['p_DM'].apply(stars)
print('\nEvaluation complete.')


## 6. Results Tables


In [ ]:
# ── Print full results per horizon ────────────────────────────────────────────
print(f'{"="*90}')
print('REALIZED VOLATILITY FORECASTING RESULTS')
print('OOS R² relative to Naive | MAPE (%) | DM-HLN test vs HAR (+ = model better)')
print(f'{"="*90}')

for h in HORIZONS:
    sub = (res_df[res_df['h'] == h]
           .set_index('Model')
           .reindex([m for m in MODEL_ORDER if m in res_df['Model'].values]))
    sub['DM_sig'] = sub['DM_vs_HAR'].map(lambda x: f'{x:+.3f}' if not pd.isna(x) else 'n/a')
    sub['DM_sig'] += sub['Sig']
    print(f'\n  h = {h}')
    print(sub[['MSE', 'MAE', 'MAPE', 'R2_OOS', 'DM_sig']]
          .to_string(float_format='{:.4f}'.format))

res_df.to_csv(OUTPUT / 'C_rv_forecasting_results.csv', index=False)
print(f'\nSaved: output/C_rv_forecasting_results.csv')

In [ ]:
# ── HAR coefficient evolution across horizons ─────────────────────────────────
# Expected Corsi (2009) pattern: beta_m dominates at long horizons
print('HAR-RV coefficient evolution (Corsi 2009 HAR model):')
print('  Expected: long-horizon persistence shifts weight to monthly component')
print()

for h in HORIZONS:
    m = models_store[(h, 'HAR')]
    c = m.params
    print(f'  h={h:2d}: const={c.iloc[0]:+.3f}  '
          f'beta_d={c.iloc[1]:+.3f}  '
          f'beta_w={c.iloc[2]:+.3f}  '
          f'beta_m={c.iloc[3]:+.3f}  '
          f'R2_train={m.rsquared:.3f}')

print()
print('HAR (h=1) full summary:')
print(models_store[(1, 'HAR')].summary())


In [ ]:
# ── M4_smooth best model (h=20) — coefficient inspection ─────────────────────
print('M4_smooth (h=20) — Best model summary:')
print(models_store[(20, 'M4_smooth')].summary())


## 7. Visualizations


In [ ]:
# ── R²_OOS heatmap ────────────────────────────────────────────────────────────
pivot = res_df.pivot(index='Model', columns='h', values='R2_OOS')
pivot = pivot.reindex([m for m in MODEL_ORDER if m in pivot.index])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=-0.05, vmax=0.95, aspect='auto')
plt.colorbar(im, ax=ax, label='R² OOS (vs Naive)')
ax.set_xticks(range(len(HORIZONS)))
ax.set_xticklabels([f'h={h}' for h in HORIZONS])
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index)

for i, mname in enumerate(pivot.index):
    for j, h in enumerate(HORIZONS):
        val = pivot.loc[mname, h]
        row = res_df[(res_df['h'] == h) & (res_df['Model'] == mname)]
        sig = row['Sig'].values[0] if len(row) else ''
        fc  = 'white' if abs(val) > 0.55 else 'black'
        ax.text(j, i, f'{val:.3f}{sig}', ha='center', va='center',
                fontsize=7.5, color=fc, weight='bold' if sig else 'normal')

# Dividers after HAR+VIX and before M4_smooth
ax.axhline(2.5, color='navy', lw=1.5, ls='--')
ax.axhline(10.5, color='darkgreen', lw=1.5, ls='--')

ax.set_title('R² OOS (relative to Naive) — stars = DM sig vs HAR\n'
             '*p<0.05  **p<0.01  ***p<0.001', pad=10)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'C_r2oos_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Forecast vs actual: HAR vs M4_smooth (h=5 and h=20) ─────────────────────
EVENTS = [
    ('2018-02-05', 'Volmageddon'),
    ('2020-02-24', 'COVID crash'),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex='col')

for col_idx, h in enumerate([5, 20]):
    df_h = df_full.join(targets[h].rename('target')).dropna()
    n_tr = int(len(df_h) * SPLIT)
    dates_te = df_h.index[n_tr:]
    p = preds_store[h]
    yte_h = p['y_true']

    ax0 = axes[0, col_idx]
    ax1 = axes[1, col_idx]

    ax0.plot(dates_te, yte_h,        label='Actual',     color='black',    lw=1.2)
    ax0.plot(dates_te, p['HAR'],      label='HAR',        color='steelblue',lw=1.0, alpha=0.85)
    ax0.plot(dates_te, p['M4_smooth'],label='M4_smooth',  color='tomato',   lw=1.0, alpha=0.9)
    ax0.set_title(f'h={h}: Forecast comparison (OOS)')
    ax0.set_ylabel('log-RV')
    ax0.legend(fontsize=8, loc='upper left')

    e_har_h = yte_h - p['HAR']
    e_m4s_h = yte_h - p['M4_smooth']
    ax1.plot(dates_te, e_har_h,  color='steelblue', lw=0.8, alpha=0.75, label='HAR error')
    ax1.plot(dates_te, e_m4s_h, color='tomato',    lw=0.8, alpha=0.75, label='M4_smooth error')
    ax1.axhline(0, color='black', lw=0.8, ls='--')
    ax1.set_ylabel('Forecast error')
    ax1.legend(fontsize=8, loc='upper left')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    for dt_str, lbl in EVENTS:
        ts = pd.Timestamp(dt_str)
        if dates_te[0] <= ts <= dates_te[-1]:
            for ax in [ax0, ax1]:
                ax.axvline(ts, color='purple', lw=1.2, ls=':', alpha=0.8)
            ax0.text(ts, ax0.get_ylim()[1] * 0.92, lbl,
                     fontsize=7, color='purple', rotation=90, va='top')

fig.suptitle('Realized Volatility Forecasting: HAR vs M4_smooth (OOS)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'C_forecast_comparison.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── Lambda_stress time series: smooth regime indicator ────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(vix_s.loc[START:END].index,
             vix_s.loc[START:END].values * 100,
             color='steelblue', lw=0.8, label='VIX (%)')
axes[0].set_ylabel('VIX (%)')
axes[0].legend(loc='upper left')
axes[0].set_title('VIX and smooth regime weight lambda_t')

axes[1].fill_between(lambda_stress.index, lambda_stress.values,
                     alpha=0.4, color='tomato', label='lambda_stress')
axes[1].plot(lambda_stress.index, lambda_stress.values,
             color='darkred', lw=0.8)
axes[1].axhline(0.5, color='black', lw=0.8, ls='--', alpha=0.5)
axes[1].set_ylabel('lambda_t (stress weight)')
axes[1].set_ylim(0, 1)
axes[1].legend(loc='upper left')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

for dt_str, lbl in EVENTS:
    for ax in axes:
        ax.axvline(pd.Timestamp(dt_str), color='purple', lw=1, ls=':', alpha=0.8)

plt.tight_layout()
plt.savefig(PLOT_DIR / 'C_lambda_stress.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
# ── MSE ratio plot (model MSE / HAR MSE) across horizons ─────────────────────
har_mse = {}
for h in HORIZONS:
    row = res_df[(res_df['h'] == h) & (res_df['Model'] == 'HAR')]
    har_mse[h] = row['MSE'].values[0]

res_df['MSE_ratio'] = res_df.apply(
    lambda r: r['MSE'] / har_mse[r['h']], axis=1
)

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.22
x = np.arange(len(MODEL_ORDER))
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, h in enumerate(HORIZONS):
    vals = []
    for m in MODEL_ORDER:
        row = res_df[(res_df['h'] == h) & (res_df['Model'] == m)]
        vals.append(row['MSE_ratio'].values[0] if len(row) else 1.0)
    ax.bar(x + (i - 1) * width, vals, width, label=f'h={h}',
           color=colors[i], alpha=0.8, edgecolor='white')

ax.axhline(1.0, color='black', lw=1.5, ls='--', label='HAR (= 1.0)')
ax.axhline(0.9, color='gray',  lw=0.8, ls=':')
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('MSE / MSE_HAR  (< 1 = better than HAR)')
ax.set_title('MSE ratio relative to HAR baseline — by model and horizon')
ax.legend()
ax.set_ylim(0.5, 1.5)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'C_mse_ratio_bar.png', dpi=130, bbox_inches='tight')
plt.show()


## 8. Key Findings

### 1. HAR as baseline (Corsi 2009)
The HAR-RV model is the natural benchmark for realized volatility forecasting due to its parsimony and strong theoretical grounding in heterogeneous agents (Müller et al. 1997). Coefficient evolution confirms Corsi's long-memory analogy: at short horizons $\beta_d$ dominates; at h=20 the monthly component $\beta_m$ is the strongest predictor.

### 2. HAR+VIX as secondary benchmark
Adding the contemporaneous VIX — as a market-consensus forward-looking volatility measure — provides incremental lift, particularly at h=5. This justifies separating the benchmark into (1) HAR-only and (2) HAR+VIX before attributing gains to SSVI features.

### 3. M1 confirms alpha ≈ integrated variance
The SSVI level parameter $\alpha$ enters significantly (p < 0.01) in M1 across all horizons. This is consistent with the Heston ↔ SSVI mapping: $\alpha \approx \log(\theta_T)$ represents the log-integrated variance, a direct forward-looking RV proxy embedded in the surface.

### 4. M4_smooth — best at h=20 (exponential regime)
The smooth regime weighting $\lambda_t = \sigma(10 \cdot (\text{VIX\_pct}_t - 0.70))$ allows SSVI parameters to have state-dependent predictive coefficients. At h=20, M4_smooth achieves R²_OOS ≈ 0.868 (DM p < 0.0001 vs HAR): the vol surface curvature parameters $\eta, \gamma$ carry significantly more predictive information during stress regimes.

### 5. F3_ATM — portable benchmark
F3_ATM replaces VIX with SSVI-derived ATM IV $= e^{\alpha/2} T^{(\beta-1)/2}$ (Gatheral & Jacquier 2014, eq. k=0 special case). Performance is competitive with F3_VIX (which uses market VIX directly), making F3_ATM applicable to any asset with a calibrated SSVI surface — not just equity indices with exchange-published VIX equivalents.

### 6. Near-arbitrage stress interaction (M4+int)
The interaction term $\log(\text{ATM\_IV}) \times \log(1 + \text{max\_cond1})$ captures episodes where the vol surface approaches the butterfly arbitrage boundary simultaneously with high vol levels. This interaction significantly improves h=5 predictions (DM = −8.75***, vs M4_smooth), consistent with NB08.5 findings: near-arbitrage stress in the SSVI surface is a leading indicator of realized volatility jumps.

### References
- Corsi, F. (2009). A simple approximate long-memory model of realized volatility. *JFEC*, 7(2), 174–196.
- Gatheral, J. & Jacquier, A. (2014). Arbitrage-free SVI volatility surfaces. *Quantitative Finance*, 14(1), 59–71.
- Diebold, F. & Mariano, R. (1995). Comparing predictive accuracy. *JBES*, 13(3), 253–263.
- Harvey, T., Leybourne, S. & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *IJF*, 13(2), 281–291.
- Andersen, T., Bollerslev, T., Diebold, F. & Labys, P. (2003). Modeling and forecasting realized volatility. *Econometrica*, 71(2), 579–625.
- Andres, H., Boumezoued, A. & Jourdain, B. (2025). The implied volatility surface (also) is path-dependent. *arXiv v3*.


## 9. Model Formulas

Complete mathematical specification of every model estimated in Section 5.
All models are estimated by OLS on log-transformed realized volatility.
Features at time $t$ only; target = $h$-step-ahead mean log-RV.

---

### Notation

| Symbol | Definition |
|--------|-----------|
| $y_t^{(h)}$ | $\frac{1}{h}\sum_{i=1}^{h}\log RV_{t+i}$ — target at horizon $h$ |
| $RV_t^{(d)}$ | $\log(\sqrt{252\,r_t^2})$ — daily log-RV |
| $RV_t^{(w)}$ | $\log\!\left(\frac{1}{5}\sum_{j=0}^{4}\sqrt{252\,r_{t-j}^2}\right)$ — weekly log-RV |
| $RV_t^{(m)}$ | $\log\!\left(\frac{1}{22}\sum_{j=0}^{21}\sqrt{252\,r_{t-j}^2}\right)$ — monthly log-RV |
| $\theta_T$ | $e^{\alpha_t}\,T^{\beta_t}$ — SSVI ATM total variance at maturity $T$ |
| $\sigma_{ATM}$ | $\sqrt{\theta_{T_0}/T_0} = e^{\alpha_t/2}\,T_0^{(\beta_t-1)/2}$ — 1Q ATM IV ($T_0=0.25$) |
| $\Delta\theta_t$ | $\theta_t - \theta_{t-1}$ — first difference of SSVI parameter $\theta$ |
| $\lambda_t$ | $\sigma\!\left(10(\mathrm{VIX\text{-}pct}_t - 0.70)\right)$ — smooth stress weight |
| $I_t$ | $\mathbf{1}\{\lambda_t > 0.5\}$ — binary hard stress indicator |
| $\kappa_t$ | $\log(1+\text{max\_cond1}_t)$ — butterfly-proximity (stress) |
| $\alpha,\beta,\rho,\eta,\gamma$ | SSVI parameters calibrated daily |

---

### Baseline models

**Naive (random walk)**
$$\hat{y}_t^{(h)} = RV_t^{(d)}$$

**HAR — primary baseline** (Corsi 2009)
$$y_t^{(h)} = c + \beta_d\,RV_t^{(d)} + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)} + \varepsilon_t$$

**HAR+VIX — secondary benchmark**
$$y_t^{(h)} = c + \beta_d\,RV_t^{(d)} + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)} + \beta_v\,\mathrm{VIX}_t + \varepsilon_t$$

---

### Portfolio models (SSVI-only, portable to any asset)

**F3_ATM — HAR augmented with SSVI ATM implied volatility**
$$y_t^{(h)} = c + \beta_d\,RV_t^{(d)} + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)}
+ \beta_{\sigma}\log\sigma_{ATM,t} + \varepsilon_t$$

Derivation: $\log\sigma_{ATM,t} = \tfrac{\alpha_t}{2} + \tfrac{\beta_t - 1}{2}\log T_0$. Entirely determined by $(\alpha_t, \beta_t)$ — no market price of the underlying needed after SSVI calibration.

**HAR_rhoJ — HAR + rho-based jump signal** (from NB08)
$$y_t^{(h)} = c + \beta_d\,RV_t^{(d)} + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)}
+ \beta_{|\Delta\rho|}\,|\Delta\rho_t| + \varepsilon_t$$

$|\Delta\rho_t| = |\rho_t - \rho_{t-1}|$ captures abrupt daily shifts in the leverage/skew parameter of the SSVI surface. Large values mark sudden repricing of tail risk — a model-free proxy for jump arrival (cf. Aït-Sahalia & Jacod 2014). Addresses the underrepresentation of $\rho$ dynamics in models that use only the $\rho$ *level*: the level enters M2/M4, but here the *change rate* provides independent predictive information at short horizons.

**M_dSSVI — HAR + first-differences of all SSVI parameters** (Andres et al. 2025, NB08)
$$y_t^{(h)} = c + \sum_{j\in\{d,w,m\}}\beta_j\,RV_t^{(j)}
+ \sum_{\theta\in\Theta}\delta_\theta\,\Delta\theta_t + \varepsilon_t$$

where $\Theta = \{\alpha,\beta,\rho,\eta,\gamma\}$ and $\Delta\theta_t = \theta_t - \theta_{t-1}$.  
This model captures the *velocity* of surface change rather than its level. Consistent with Andres et al. (2025): the implied volatility surface is path-dependent — its dynamics convey information beyond its current state. Best overall model at h=20 in NB08 (R²_OOS = 0.8412).

**M1 — HAR + SSVI level**
$$y_t^{(h)} = c + \beta_d\,RV_t^{(d)} + \beta_w\,RV_t^{(w)} + \beta_m\,RV_t^{(m)}
+ \beta_\alpha\,\alpha_t + \varepsilon_t$$

**M2 — HAR + term-structure & skew**
$$y_t^{(h)} = c + \sum_{j \in \{d,w,m\}}\beta_j\,RV_t^{(j)}
+ \beta_\beta\,\beta_t + \beta_\rho\,\rho_t + \varepsilon_t$$

$\beta_t$ captures how total variance scales with maturity; $\rho_t$ encodes leverage / crash-fear skew.

**M3 — HAR + curvature**
$$y_t^{(h)} = c + \sum_{j \in \{d,w,m\}}\beta_j\,RV_t^{(j)}
+ \beta_\eta\,\eta_t + \beta_\gamma\,\gamma_t + \varepsilon_t$$

$\eta_t, \gamma_t$ parametrize the smile's amplitude and decay — tail-risk pricing.

**M4 — HAR + full SSVI surface**
$$y_t^{(h)} = c + \sum_{j \in \{d,w,m\}}\beta_j\,RV_t^{(j)}
+ \beta_\alpha\,\alpha_t + \beta_\beta\,\beta_t + \beta_\rho\,\rho_t
+ \beta_\eta\,\eta_t + \beta_\gamma\,\gamma_t + \varepsilon_t$$

---

### Augmented models

**M4_thresh — Hard stress indicator** (TAR analogy, Hansen 1999)

$$y_t^{(h)} = c + \sum_{j}\beta_j\,RV_t^{(j)}
+ \sum_{\theta\in\Theta}\left(\beta_\theta\,\theta_t + \delta_\theta\,\theta_t\cdot I_t\right)
+ \gamma_I\,I_t + \varepsilon_t$$

where $I_t = \mathbf{1}\{\lambda_t > 0.5\} = \mathbf{1}\{\mathrm{VIX\text{-}pct}_t > 0.70\}$.

This is the discrete-threshold analogue of M4_smooth: SSVI coefficients switch abruptly between a calm-regime slope $\beta_\theta$ and a stressed-regime slope $\beta_\theta + \delta_\theta$ at the 70th VIX-percentile boundary. Directly comparable to M4_smooth, which uses the same threshold but via a smooth sigmoid transition.

**M4_smooth — Smooth regime weighting** (Teräsvirta 1994 STAR analogy)

$$y_t^{(h)} = c + \sum_{j \in \{d,w,m\}}\beta_j\,RV_t^{(j)}
+ \sum_{\theta\in\Theta}\left(\beta_\theta\,\theta_t + \delta_\theta\,\theta_t\cdot\lambda_t\right)
+ \gamma_\lambda\,\lambda_t + \varepsilon_t$$

where $\Theta = \{\alpha,\beta,\rho,\eta,\gamma\}$ and
$$\lambda_t = \frac{1}{1 + e^{-10\,(\mathrm{VIX\text{-}pct}_t - 0.70)}} \in [0,1]$$

The smooth stress weight $\lambda_t$ allows each SSVI coefficient to transition continuously between a calm-regime slope $\beta_\theta$ and a stressed-regime slope $\beta_\theta + \delta_\theta$, avoiding the discontinuity of hard threshold models (Hansen 1999).

**M5 — M4 + Nonlinear SSVI terms**
$$y_t^{(h)} = c + \sum_{j}\beta_j\,RV_t^{(j)}
+ \sum_{\theta\in\Theta}\beta_\theta\,\theta_t
+ \beta_{\beta^2}\,\beta_t^2 + \beta_{\alpha\beta}\,\alpha_t\beta_t + \varepsilon_t$$

$\beta_t^2$ captures convex maturity-term-structure effects; $\alpha_t\beta_t$ interacts the level and slope of total variance.

**M4+int — Near-arbitrage stress interaction**
$$y_t^{(h)} = \text{M4\_smooth}_t
+ \beta_{\kappa}\,\log\sigma_{ATM,t} \cdot \kappa_t + \varepsilon_t$$

where $\kappa_t = \log(1 + \text{max\_cond1}_t)$.

Geometric interpretation of $\kappa_t$: the butterfly no-arbitrage condition requires
$g(k,T) \geq 0$ for all strikes, where $g$ is a curvature measure of the SSVI surface.
`max_cond1` $\to 0$ means the surface is operating at the maximum curvature permitted
without generating negative risk-neutral densities — the vol surface is at its
structurally tightest. The interaction $\log\sigma_{ATM}\cdot\kappa$ multiplies the
vol *level* by this tightness, identifying episodes where both the surface level and its
curvature stress peak simultaneously — the empirical precursor to realized vol jumps.

---

### Evaluation metrics

$$R^2_{OOS} = 1 - \frac{\sum_t (y_t - \hat{y}_t)^2}{\sum_t (y_t - RV_{t-1}^{(d)})^2}$$

$$\mathrm{RMSE} = \sqrt{\frac{1}{T_{OOS}}\sum_t (y_t - \hat{y}_t)^2}$$

$$\mathrm{MAE} = \frac{1}{T_{OOS}}\sum_t |y_t - \hat{y}_t|$$

$$\mathrm{MAPE} = \frac{1}{T_{OOS}}\sum_t \frac{|y_t - \hat{y}_t|}{|y_t|} \times 100\%$$

**DM-HLN statistic** (H₀: equal predictive accuracy vs HAR):
$$DM = \frac{\bar{d}}{\sqrt{\hat{V}(d)\,\kappa/T}} \xrightarrow{d} \mathcal{N}(0,1)$$

where $d_t = e_{\mathrm{HAR},t}^2 - e_{\mathrm{model},t}^2$,
$\hat{V}(d)$ is the Bartlett HAC variance with $h-1$ lags, and
$$\kappa = \frac{T + 1 - 2h + h(h-1)/T}{T}$$
is the Harvey–Leybourne–Newbold small-sample correction factor.
